# 5a HMM 训练
---

In [16]:
import json
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import entropy
from hmmlearn.hmm import GaussianHMM

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
np.set_printoptions(precision=4, suppress=True)

DATA_DIR = Path("data")
RANDOM_STATES = [0, 7, 17, 42, 123]
K_GRID = [2, 3, 4]

---
## 2 读取因子矩阵、splits、Step 4 锁定的变换函数

In [17]:
factors = pd.read_parquet(DATA_DIR / "spy_factors_step2.parquet")
with open(DATA_DIR / "splits.json") as f:
    splits = json.load(f)

train_start, train_end = splits["train"]["start"], splits["train"]["end"]
val_start, val_end = splits["val"]["start"], splits["val"]["end"]
test_start, test_end = splits["test"]["start"], splits["test"]["end"]

X_HMM_COLS = ["RV_open30", "RV_close30", "Jump", "VR_5_1",'Ret_1d','Mom_20d','Mom_60d']

def log_scaled(x, scale):
    return np.log1p(x * scale)

X_raw = factors[X_HMM_COLS].copy()
X_tfm = pd.DataFrame(index=X_raw.index, columns=X_HMM_COLS, dtype=float)
X_tfm["RV_open30"]  = log_scaled(X_raw["RV_open30"],  1e5)
X_tfm["RV_close30"] = log_scaled(X_raw["RV_close30"], 1e5)
X_tfm["Jump"]       = log_scaled(X_raw["Jump"],       1e5)
X_tfm["VR_5_1"]     = X_raw["VR_5_1"]
X_tfm["Ret_1d"]       = X_raw["Ret_1d"]
X_tfm["Mom_20d"]       = X_raw["Mom_20d"]
X_tfm["Mom_60d"]       = X_raw["Mom_60d"]
X_tfm=X_tfm.dropna()
print(X_tfm.head())

print("factors shape:", factors.shape)
print("train:", train_start, "→", train_end)
print("val:  ", val_start,   "→", val_end)
print("test: ", test_start,  "→", test_end)
print("\nX_tfm 在 train 段的描述统计:")
print(X_tfm.loc[train_start:train_end].describe())
print("\nNaN 计数(全样本):")
print(X_tfm.isna().sum())

            RV_open30  RV_close30   Jump  VR_5_1  Ret_1d  Mom_20d  Mom_60d
date                                                                      
2015-03-31     0.5749      0.6319 0.5274  0.9547 -0.0088  -0.0225   0.0050
2015-04-01     0.6903      0.5351 0.6006  1.0592 -0.0033  -0.0215   0.0194
2015-04-02     0.6784      0.2674 0.3122  1.2574  0.0030  -0.0196   0.0323
2015-04-06     0.5186      0.1088 1.4381  0.1401  0.0070   0.0017   0.0268
2015-04-07     0.3003      0.3363 0.2883  1.0793 -0.0028  -0.0054   0.0065
factors shape: (2745, 25)
train: 2017-01-05 → 2023-04-25
val:   2023-04-26 → 2024-08-27
test:  2024-08-28 → 2025-12-31

X_tfm 在 train 段的描述统计:
       RV_open30  RV_close30      Jump    VR_5_1    Ret_1d   Mom_20d   Mom_60d
count  1569.0000   1569.0000 1569.0000 1569.0000 1569.0000 1569.0000 1569.0000
mean      0.5703      0.4664    0.2563    0.9302    0.0004    0.0076    0.0234
std       0.5082      0.5191    0.3975    0.2290    0.0115    0.0501    0.0725
min       0.0305 

---
## 3  Expanding z-score 标准化

In [18]:
def expanding_zscore(series, min_periods=60):
    mean = series.expanding(min_periods=min_periods).mean().shift(1)
    std  = series.expanding(min_periods=min_periods).std().shift(1)
    return (series - mean) / std

X_std = pd.DataFrame(index=X_tfm.index, columns=X_HMM_COLS, dtype=float)
for col in X_HMM_COLS:
    X_std[col] = expanding_zscore(X_tfm[col])

X_std_train = X_std.loc[train_start:train_end].dropna()
X_std_val   = X_std.loc[val_start:val_end].dropna()
X_std_test  = X_std.loc[test_start:test_end].dropna()

print("X_std_train shape:", X_std_train.shape)
print("X_std_val   shape:", X_std_val.shape)
print("X_std_test  shape:", X_std_test.shape)
print("\nPost-standardization train segment statistics (mean should be ≈ 0, std should be ≈ 1):")
print(X_std_train.describe().loc[["mean", "std", "min", "max"]])

X_std.to_parquet(DATA_DIR / "X_HMM_standardized.parquet")
print("\nsaved: data/X_HMM_standardized.parquet")

X_std_train shape: (1569, 7)
X_std_val   shape: (336, 7)
X_std_test  shape: (337, 7)

Post-standardization train segment statistics (mean should be ≈ 0, std should be ≈ 1):
      RV_open30  RV_close30    Jump  VR_5_1  Ret_1d  Mom_20d  Mom_60d
mean     0.2709      0.2544  0.1209  0.0653  0.0001   0.0127   0.0557
std      1.2239      1.2114  1.2142  1.0347  1.1976   1.2757   1.2571
min     -1.1058     -0.8205 -0.7472 -4.1562 -7.1502 -10.9985  -7.2996
max      9.4697      9.7818  9.4008  2.8775  9.6296   4.4723   5.0067

saved: data/X_HMM_standardized.parquet


---
## 4 在 train 段拟合 K ∈ {2,3,4},每 K 用 5 个 random_state

In [19]:
def fit_hmm_multi_seed(X, K, seeds, n_iter=200, tol=1e-4):
    best_model, best_ll = None, -np.inf
    all_lls = []
    for s in seeds:
        m = GaussianHMM(
            n_components=K,
            covariance_type="full",
            n_iter=n_iter,
            tol=tol,
            random_state=s,
        )
        try:
            m.fit(X)
            ll = m.score(X)
            all_lls.append((s, ll, m.monitor_.converged))
            if ll > best_ll:
                best_ll, best_model = ll, m
        except Exception as e:
            all_lls.append((s, np.nan, False))
    return best_model, best_ll, all_lls

X_train_arr = X_std_train.values
n_train = len(X_train_arr)
d = X_train_arr.shape[1]

results = {}
for K in K_GRID:
    model, ll, all_lls = fit_hmm_multi_seed(X_train_arr, K, RANDOM_STATES)
    n_params = K * (K - 1) + K * d + K * d * (d + 1) / 2
    bic = -2 * ll + n_params * np.log(n_train)
    aic = -2 * ll + 2 * n_params
    results[K] = {
        "model": model, "ll": ll, "bic": bic, "aic": aic,
        "n_params": n_params, "all_seeds": all_lls,
    }

print(f"{'K':>3} {'logL':>12} {'BIC':>12} {'AIC':>12} {'n_params':>10}")
for K in K_GRID:
    r = results[K]
    print(f"{K:>3} {r['ll']:>12.2f} {r['bic']:>12.2f} {r['aic']:>12.2f} {r['n_params']:>10.0f}")

print("\nlogL across 5 seeds for each K (check EM convergence stability):")
for K in K_GRID:
    print(f"  K={K}:")
    for s, ll, conv in results[K]["all_seeds"]:
        flag = "✓" if conv else "✗"
        print(f"    seed={s:>4}  logL={ll:>12.2f}  converged={flag}")

print("\nΔBIC (K vs K-1):")
for K in K_GRID[1:]:
    d_bic = results[K-1]["bic"] - results[K]["bic"]
    print(f"  K={K-1} → K={K}: ΔBIC = {d_bic:+.2f}  (accept higher K only if >30)")

  File "d:\python\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
        "wmic CPU Get NumberOfCores /Format:csv".split(),
        capture_output=True,
        text=True,
    )
  File "d:\python\Lib\subprocess.py", line 556, in run
    with Popen(*popenargs, **kwargs) as process:
         ~~~~~^^^^^^^^^^^^^^^^^^^^^^
  File "d:\python\Lib\subprocess.py", line 1038, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                        pass_fds, cwd, env,
                        ^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
                        gid, gids, uid, umask,
                        ^^^^^^^^^^^^^^^^^^^^^^
                        start_new_session, process_group)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\python\Lib\subprocess.py", line 1550, in _execute_child
    hp, ht, pid, tid = _wi

  K         logL          BIC          AIC   n_params
  2    -12752.22     26034.23     25648.44         72
  3    -11910.36     24637.47     24042.71        111
  4    -11542.88     24204.20     23389.75        152

logL across 5 seeds for each K (check EM convergence stability):
  K=2:
    seed=   0  logL=   -12752.22  converged=✓
    seed=   7  logL=   -12752.22  converged=✓
    seed=  17  logL=   -12752.22  converged=✓
    seed=  42  logL=   -12752.22  converged=✓
    seed= 123  logL=   -12752.22  converged=✓
  K=3:
    seed=   0  logL=   -11910.36  converged=✓
    seed=   7  logL=   -12013.78  converged=✓
    seed=  17  logL=   -12013.78  converged=✓
    seed=  42  logL=   -12013.78  converged=✓
    seed= 123  logL=   -12013.78  converged=✓
  K=4:
    seed=   0  logL=   -11549.50  converged=✓
    seed=   7  logL=   -11545.95  converged=✓
    seed=  17  logL=   -11542.88  converged=✓
    seed=  42  logL=   -11547.45  converged=✓
    seed= 123  logL=   -11545.95  converged=✓

ΔBIC (

---
## 5 转移矩阵、自转移概率、平均持续时长(描述性,不做决策)

In [20]:
def regime_durations(labels):
    durations = {}
    if len(labels) == 0:
        return durations
    cur, count = labels[0], 1
    runs = []
    for x in labels[1:]:
        if x == cur:
            count += 1
        else:
            runs.append((cur, count))
            cur, count = x, 1
    runs.append((cur, count))
    for r, c in runs:
        durations.setdefault(r, []).append(c)
    return durations

for K in K_GRID:
    print(f"\n{'='*60}")
    print(f"K = {K}")
    print(f"{'='*60}")
    model = results[K]["model"]
    A = model.transmat_
    pi = model.get_stationary_distribution()
    labels = model.predict(X_train_arr)
    counts = pd.Series(labels).value_counts().sort_index()
    
    print(f"\nTransition matrix A:")
    print(pd.DataFrame(A, index=[f"from_{i}" for i in range(K)],
                       columns=[f"to_{i}" for i in range(K)]))
    print(f"\nStationary distribution π:")
    print(pd.Series(pi, index=[f"r{i}" for i in range(K)]))
    print(f"\nSelf-transition vs marginal probability (self-transition should exceed marginal by > 0.1):")
    for i in range(K):
        marg = counts[i] / counts.sum() if i in counts.index else 0
        print(f"  r{i}: A[{i},{i}]={A[i,i]:.4f}  marginal={marg:.4f}  diff={A[i,i]-marg:+.4f}")
    
    durs = regime_durations(labels)
    print(f"\nRegime durations under Viterbi labels (mean should be ≥ 3 days):")
    for r in sorted(durs.keys()):
        arr = np.array(durs[r])
        print(f"  r{r}: n_runs={len(arr):>4}  mean={arr.mean():.2f}  median={np.median(arr):.0f}  max={arr.max()}")
    
    print(f"\nSample count / proportion per regime under Viterbi (train segment):")
    for i in range(K):
        n = (labels == i).sum()
        print(f"  r{i}: n={n:>5}  pct={n/len(labels)*100:.2f}%")


K = 2

Transition matrix A:
         to_0   to_1
from_0 0.8760 0.1240
from_1 0.1116 0.8884

Stationary distribution π:
r0   0.4737
r1   0.5263
dtype: float64

Self-transition vs marginal probability (self-transition should exceed marginal by > 0.1):
  r0: A[0,0]=0.8760  marginal=0.4685  diff=+0.4075
  r1: A[1,1]=0.8884  marginal=0.5315  diff=+0.3568

Regime durations under Viterbi labels (mean should be ≥ 3 days):
  r0: n_runs=  85  mean=8.65  median=1  max=145
  r1: n_runs=  86  mean=9.70  median=6  max=56

Sample count / proportion per regime under Viterbi (train segment):
  r0: n=  735  pct=46.85%
  r1: n=  834  pct=53.15%

K = 3

Transition matrix A:
         to_0   to_1   to_2
from_0 0.5418 0.1070 0.3512
from_1 0.0432 0.8749 0.0819
from_2 0.1371 0.0785 0.7843

Stationary distribution π:
r0   0.1658
r1   0.4088
r2   0.4254
dtype: float64

Self-transition vs marginal probability (self-transition should exceed marginal by > 0.1):
  r0: A[0,0]=0.5418  marginal=0.1619  diff=+0.3799
  

---
## 6 各 regime 在 4 个因子维度上的均值与方差(可解释性)

In [21]:
for K in K_GRID:
    print(f"\n{'='*60}")
    print(f"K = {K}  Per-regime statistics across X_HMM dimensions")
    print(f"{'='*60}")
    model = results[K]["model"]
    labels = model.predict(X_train_arr)
    df = X_std_train.copy()
    df["regime"] = labels
    
    print("\nMean (standardized space):")
    print(df.groupby("regime")[X_HMM_COLS].mean())
    print("\nStd (standardized space; >1 indicates within-regime variance exceeds training mean variance):")
    print(df.groupby("regime")[X_HMM_COLS].std())
    print("\nSample count:")
    print(df.groupby("regime").size())


K = 2  Per-regime statistics across X_HMM dimensions

Mean (standardized space):
        RV_open30  RV_close30    Jump  VR_5_1  Ret_1d  Mom_20d  Mom_60d
regime                                                                 
0          1.0628      0.9719  0.7075 -0.0060 -0.1234  -0.5034  -0.5879
1         -0.4269     -0.3779 -0.3961  0.1281  0.1089   0.4675   0.6229

Std (standardized space; >1 indicates within-regime variance exceeds training mean variance):
        RV_open30  RV_close30   Jump  VR_5_1  Ret_1d  Mom_20d  Mom_60d
regime                                                                
0          1.3402      1.4403 1.5500  1.1760  1.6347   1.6389   1.5031
1          0.4429      0.2826 0.2946  0.8878  0.5653   0.5047   0.5444

Sample count:
regime
0    735
1    834
dtype: int64

K = 3  Per-regime statistics across X_HMM dimensions

Mean (standardized space):
        RV_open30  RV_close30    Jump  VR_5_1  Ret_1d  Mom_20d  Mom_60d
regime                                      

---
## 9 Lock K=2 and generate causal forward labels (train/val/test)

In [22]:
K_FINAL = 2
final_model = results[K_FINAL]["model"]

def causal_forward_labels(model, X):
    log_A = np.log(model.transmat_ + 1e-300)
    log_pi = np.log(model.startprob_ + 1e-300)
    log_B = model._compute_log_likelihood(X)
    T, K = log_B.shape
    log_alpha = np.zeros((T, K))
    log_alpha[0] = log_pi + log_B[0]
    for t in range(1, T):
        log_alpha[t] = log_B[t] + np.array([
            np.logaddexp.reduce(log_alpha[t-1] + log_A[:, j])
            for j in range(K)
        ])
    log_alpha -= log_alpha.max(axis=1, keepdims=True)
    alpha = np.exp(log_alpha)
    alpha /= alpha.sum(axis=1, keepdims=True)
    labels = alpha.argmax(axis=1)
    return labels, alpha

X_full = X_std.dropna()
labels_fwd, posterior = causal_forward_labels(final_model, X_full.values)
labels_vit_train = final_model.predict(X_std_train.values)

regime_forward = pd.Series(labels_fwd, index=X_full.index, name="regime")
regime_posterior = pd.DataFrame(posterior, index=X_full.index,
                                columns=[f"p_r{i}" for i in range(K_FINAL)])
regime_viterbi_train = pd.Series(labels_vit_train, index=X_std_train.index, name="regime")

print(f"Forward labels generated: {len(regime_forward)} rows")
print(f"\nForward label distribution per split:")
for name, (s, e) in [("train", (train_start, train_end)),
                     ("val",   (val_start,   val_end)),
                     ("test",  (test_start,  test_end))]:
    seg = regime_forward.loc[s:e]
    counts = seg.value_counts().sort_index()
    print(f"  {name} (n={len(seg)}): " +
          "  ".join([f"r{i}={counts.get(i,0)} ({counts.get(i,0)/len(seg)*100:.1f}%)"
                     for i in range(K_FINAL)]))

agree = (regime_forward.loc[X_std_train.index].values == labels_vit_train).mean()
print(f"\nForward vs Viterbi agreement on train: {agree:.4f} (should be >= 0.70)")

Forward labels generated: 2625 rows

Forward label distribution per split:
  train (n=1569): r0=738 (47.0%)  r1=831 (53.0%)
  val (n=336): r0=54 (16.1%)  r1=282 (83.9%)
  test (n=337): r0=124 (36.8%)  r1=213 (63.2%)

Forward vs Viterbi agreement on train: 0.9688 (should be >= 0.70)


In [23]:
regime_forward.to_frame().to_parquet(DATA_DIR / "regime_forward.parquet")
regime_posterior.to_parquet(DATA_DIR / "regime_posterior.parquet")
regime_viterbi_train.to_frame().to_parquet(DATA_DIR / "regime_viterbi.parquet")